# NumPy and Pandas Transforms

TimeDataModel provides clean patterns for transforming time series data using numpy and pandas. Every `TimeSeries` and `TimeSeriesTable` exposes `.arr` (numpy array) and `.df` (pandas DataFrame) properties, plus dedicated methods for writing results back. This keeps your domain model structured while letting you leverage the full scientific Python ecosystem.

In [1]:
from datetime import datetime, timedelta, timezone

import numpy as np

import timedatamodel as tdm

base = datetime(2024, 1, 15, tzinfo=timezone.utc)
timestamps = [base + timedelta(hours=i) for i in range(24)]

ts = tdm.TimeSeries(
    tdm.Frequency.PT1H,
    timestamps=timestamps,
    values=[
        120.0, 115.0, 108.0, 105.0, 102.0, 100.0,
        110.0, 135.0, 160.0, 175.0, 180.0, 178.0,
        172.0, 170.0, 168.0, 165.0, 175.0, 190.0,
        200.0, 195.0, 180.0, 165.0, 145.0, 130.0,
    ],
    name="power",
    unit="MW",
    data_type=tdm.DataType.MEASUREMENT,
)

ModuleNotFoundError: No module named 'timedatamodel'

## The `.arr` and `.df` properties

Every `TimeSeries` has two shorthand properties for quick access to the underlying data:
- `ts.arr` — returns a numpy `ndarray` (same as `ts.to_numpy()`)
- `ts.df` — returns a pandas `DataFrame` (same as `ts.to_pandas_dataframe()`)

In [2]:
ts.arr

NameError: name 'ts' is not defined

In [3]:
ts.df.head()

NameError: name 'ts' is not defined

## Pattern 1: `apply_numpy(func)`

Pass a function that receives a numpy array and returns a numpy array. Timestamps, frequency, and all metadata are preserved automatically. The output array must have the same length as the input.

In [4]:
normalized = ts.apply_numpy(lambda arr: (arr - arr.mean()) / arr.std())
normalized

NameError: name 'ts' is not defined

In [5]:
cumulative = ts.apply_numpy(np.cumsum)
cumulative.head(6)

NameError: name 'ts' is not defined

In [6]:
clipped = ts.apply_numpy(lambda arr: np.clip(arr, 110, 180))
clipped

NameError: name 'ts' is not defined

## Pattern 2: `apply_pandas(func)`

Pass a function that receives a pandas DataFrame and returns a pandas DataFrame. This lets you use the full pandas API — rolling windows, resampling, interpolation, and more.

In [7]:
rolling_mean = ts.apply_pandas(lambda df: df.rolling(6, min_periods=1).mean())
rolling_mean

NameError: name 'ts' is not defined

In [8]:
diff = ts.apply_pandas(lambda df: df.diff())
diff

NameError: name 'ts' is not defined

In [9]:
pct_change = ts.apply_pandas(lambda df: df.pct_change() * 100)
pct_change.head(6)

NameError: name 'ts' is not defined

## Pattern 3: One-liner round-trips with `update_arr()` and `update_df()`

Combine `.arr` / `.df` with `update_arr()` / `update_df()` to transform data in a single expression. The result is a new `TimeSeries` with all metadata preserved.

In [10]:
ts.update_arr(ts.arr.clip(110, 180))

NameError: name 'ts' is not defined

In [11]:
ts.update_df(ts.df.resample("3h").mean())

NameError: name 'ts' is not defined

In [12]:
ts.update_df(ts.df.diff())

NameError: name 'ts' is not defined

In [13]:
ts.update_arr(np.cumsum(ts.arr))

NameError: name 'ts' is not defined

## Pattern 4: Manual numpy round-trip

For transformations where the output shape differs from the input, export to numpy, transform freely, and construct a new `TimeSeries`.

In [14]:
arr = ts.to_numpy()
print(f"Type:  {type(arr)}")
print(f"Shape: {arr.shape}")
print(f"Mean:  {arr.mean():.1f} MW")

NameError: name 'ts' is not defined

In [15]:
window = 3
smoothed_arr = np.convolve(arr, np.ones(window) / window, mode="valid")
smoothed_timestamps = timestamps[window - 1 :]

ts_smoothed = tdm.TimeSeries(
    tdm.Frequency.PT1H,
    timestamps=smoothed_timestamps,
    values=smoothed_arr.tolist(),
    name=ts.name,
    unit=ts.unit,
    data_type=ts.data_type,
)
print(f"Original length: {len(ts)}, Smoothed length: {len(ts_smoothed)}")
ts_smoothed.head(6)

NameError: name 'arr' is not defined

## Pattern 5: Manual pandas round-trip

For multi-step pandas workflows where a one-liner would be hard to read, break it into separate steps.

In [16]:
df = ts.to_pandas_dataframe()
df.head()

NameError: name 'ts' is not defined

In [17]:
df_resampled = df.resample("3h").mean()

ts_resampled = ts.update_from_pandas(df_resampled)
print(f"Original:  {len(ts)} points")
print(f"Resampled: {len(ts_resampled)} points")
print(f"Unit preserved: {ts_resampled.unit}")
ts_resampled

NameError: name 'df' is not defined

In [18]:
df_ewm = df.ewm(span=6).mean()

ts_ewm = ts.update_from_pandas(df_ewm)
ts_ewm

NameError: name 'df' is not defined

## Transforms on TimeSeriesTable

All patterns — `apply_*`, `update_arr()`, `update_df()`, `.arr`, `.df` — also work on `TimeSeriesTable`, applying across all columns.

In [19]:
rng = np.random.default_rng(42)

table = tdm.TimeSeriesTable(
    tdm.Frequency.PT1H,
    timestamps=timestamps,
    values=np.column_stack([
        80 + 40 * np.sin(np.linspace(0, 2 * np.pi, 24)) + rng.normal(0, 5, 24),
        np.clip(60 * np.sin(np.linspace(-0.5, np.pi + 0.5, 24)), 0, None),
        50 + rng.normal(0, 3, 24),
    ]),
    names=["wind", "solar", "hydro"],
    units=["MW", "MW", "MW"],
)
table

NameError: name 'tdm' is not defined

In [20]:
table_rolling = table.apply_pandas(lambda df: df.rolling(4, min_periods=1).mean())
table_rolling

NameError: name 'table' is not defined

In [21]:
table_norm = table.apply_numpy(
    lambda arr: (arr - arr.mean(axis=0)) / arr.std(axis=0)
)
table_norm.head(6)

NameError: name 'table' is not defined

In [22]:
table.update_df(table.df.rolling(4, min_periods=1).mean())

NameError: name 'table' is not defined

In [23]:
table.update_arr(np.clip(table.arr, 40, 120))

NameError: name 'table' is not defined

## Summary

Five patterns for transforming time series data:

| Pattern | Method | Best for |
|---------|--------|----------|
| `ts.apply_numpy(func)` | Functional | Same-length vectorized ops (normalize, cumsum) |
| `ts.apply_pandas(func)` | Functional | Rolling windows, diff, pct_change |
| `ts.update_arr(ts.arr.clip(...))` | One-liner | Quick numpy transforms via `.arr` |
| `ts.update_df(ts.df.resample(...).mean())` | One-liner | Quick pandas transforms via `.df` |
| Manual `to_numpy()` / `to_pandas_dataframe()` | Multi-step | Shape-changing ops, complex workflows |

All patterns preserve metadata. Use `.arr` / `.df` for read access and `update_arr()` / `update_df()` to write results back.

Next up: **nb_03** covers unit handling, validation, and rich metadata.